make transition matrix plots eg fig s5

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path


# =============================================================================
# INPUT / OUTPUT
# =============================================================================

INPUT_ROOT = Path("./input").resolve()

OUTPUT_DIR = Path(
    "./output/fig-s5"
).resolve()

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

P_SAMPLES_FILE = (
    INPUT_ROOT
    / "state-model"
    / "t250_k50"
    / "bmsm"
    / "lag400"
    / "P-samples-lag-400.npz"
)

TPM_OUTPUT_FILE = (
    OUTPUT_DIR
    / "transition_probability_matrix.pdf"
)

RATE_OUTPUT_FILE = (
    OUTPUT_DIR
    / "approximate_rate_matrix.pdf"
)

if not P_SAMPLES_FILE.is_file():
    raise FileNotFoundError(
        f"Could not find P-sample file:\n{P_SAMPLES_FILE}"
    )


# =============================================================================
# SETTINGS
# =============================================================================

LAGTIME_NS = 40.0

THRESHOLD_PROB = 0.01

CI = 0.95
Q_LO = 0.5 * (1.0 - CI)
Q_HI = 1.0 - Q_LO

RATE_SCALE = 1000.0  # 1/ns -> 1/us


# =============================================================================
# LOAD POSTERIOR TPM SAMPLES
# =============================================================================

with np.load(P_SAMPLES_FILE, allow_pickle=False) as data:

    print("Available arrays in P-sample file:")
    for key in data.files:
        print(
            f"  {key}: "
            f"shape={data[key].shape}, "
            f"dtype={data[key].dtype}"
        )

    # Find arrays compatible with posterior TPM samples:
    # shape = (n_samples, k, k)
    candidate_keys = []

    for key in data.files:
        arr = np.asarray(data[key])

        if (
            arr.ndim == 3
            and arr.shape[1] == arr.shape[2]
        ):
            candidate_keys.append(key)

    if len(candidate_keys) == 0:
        raise ValueError(
            "No array with shape (S, k, k) was found in:\n"
            f"{P_SAMPLES_FILE}"
        )

    if len(candidate_keys) > 1:
        raise ValueError(
            "More than one possible TPM-sample array was found.\n"
            "Candidates:\n"
            + "\n".join(
                f"  - {key}"
                for key in candidate_keys
            )
            + "\n\n"
            "The intended array must be selected explicitly."
        )

    P_samples = np.asarray(
        data[candidate_keys[0]],
        dtype=float,
    ).copy()


print(
    f"\nUsing TPM samples: {candidate_keys[0]}"
)

print(
    f"P_samples shape: {P_samples.shape}"
)


# =============================================================================
# VALIDATE TPM SAMPLES
# =============================================================================

if P_samples.ndim != 3:
    raise ValueError(
        f"P_samples must have shape (S,k,k); got {P_samples.shape}"
    )

n_samples, k, k2 = P_samples.shape

if k != k2:
    raise ValueError(
        f"TPMs must be square; got {P_samples.shape}"
    )

if np.any(~np.isfinite(P_samples)):
    raise ValueError(
        "P_samples contains non-finite values."
    )

if np.min(P_samples) < -1e-12:
    raise ValueError(
        "P_samples contains negative transition probabilities."
    )

row_sums = P_samples.sum(axis=2)

max_row_sum_error = float(
    np.max(
        np.abs(
            row_sums - 1.0
        )
    )
)

print(
    f"Number of posterior samples: {n_samples}"
)

print(
    f"Number of states: {k}"
)

print(
    f"Maximum TPM row-sum error: "
    f"{max_row_sum_error:.3e}"
)


# =============================================================================
# STATIONARY DISTRIBUTION
# =============================================================================

def stationary_prob_solve(P):
    """
    Solve

        pi = pi P
        sum(pi) = 1

    using a least-squares augmented linear system.
    """

    P = np.asarray(
        P,
        dtype=float,
    )

    k_local = P.shape[0]

    A = (
        P.T
        - np.eye(
            k_local,
            dtype=float,
        )
    )

    b = np.zeros(
        k_local,
        dtype=float,
    )

    # Replace final equation with normalization.
    A[-1, :] = 1.0
    b[-1] = 1.0

    pi, *_ = np.linalg.lstsq(
        A,
        b,
        rcond=None,
    )

    if np.any(~np.isfinite(pi)):
        raise ValueError(
            "Non-finite stationary probability encountered."
        )

    # Permit only tiny floating-point negatives.
    minimum = float(
        np.min(pi)
    )

    if minimum < -1e-10:
        raise ValueError(
            "Stationary solution contains a substantial "
            f"negative probability: {minimum:.3e}"
        )

    pi = np.maximum(
        pi,
        0.0,
    )

    total = float(
        np.sum(pi)
    )

    if total <= 0.0:
        raise ValueError(
            "Stationary distribution has non-positive total probability."
        )

    pi /= total

    return pi


stationary_prob_samples = np.empty(
    (
        n_samples,
        k,
    ),
    dtype=float,
)

for sample_index in range(
    n_samples
):
    stationary_prob_samples[
        sample_index
    ] = stationary_prob_solve(
        P_samples[
            sample_index
        ]
    )


stationary_center = np.quantile(
    stationary_prob_samples,
    0.5,
    axis=0,
)

stationary_lo = np.quantile(
    stationary_prob_samples,
    Q_LO,
    axis=0,
)

stationary_hi = np.quantile(
    stationary_prob_samples,
    Q_HI,
    axis=0,
)


# =============================================================================
# SELECT STATES ABOVE 1%
# =============================================================================

order = np.argsort(
    -stationary_center
).astype(int)

selected_states = order[
    stationary_center[
        order
    ] > THRESHOLD_PROB
].astype(int)

if selected_states.size == 0:
    raise ValueError(
        f"No states have stationary probability > {THRESHOLD_PROB:.2f}."
    )

print(
    f"\nSelected {selected_states.size} states "
    f"with stationary probability > {THRESHOLD_PROB:.2f}"
)

print(
    "State IDs, highest population first:"
)

for state in selected_states:
    print(
        f"  state {state:2d}: "
        f"{stationary_center[state]:.5f} "
        f"[{stationary_lo[state]:.5f}, "
        f"{stationary_hi[state]:.5f}]"
    )


# =============================================================================
# RESTRICT TPM SAMPLES TO SELECTED STATES
# =============================================================================

P_sub_samples = (
    P_samples[
        :,
        selected_states,
        :,
    ][
        :,
        :,
        selected_states,
    ]
)

m = int(
    P_sub_samples.shape[1]
)


# =============================================================================
# SUMMARIZE TPM ENTRIES
# =============================================================================

P_center = np.quantile(
    P_sub_samples,
    0.5,
    axis=0,
)

P_lo = np.quantile(
    P_sub_samples,
    Q_LO,
    axis=0,
)

P_hi = np.quantile(
    P_sub_samples,
    Q_HI,
    axis=0,
)

P_err_lo = np.maximum(
    P_center - P_lo,
    0.0,
)

P_err_hi = np.maximum(
    P_hi - P_center,
    0.0,
)


# =============================================================================
# TPM HEATMAP
# =============================================================================

vmax_P = float(
    np.nanpercentile(
        P_center,
        95,
    )
)

vmax_P = max(
    vmax_P,
    1e-12,
)

fig, ax = plt.subplots(
    figsize=(9, 8),
    constrained_layout=True,
)

im = ax.imshow(
    P_center,
    cmap="inferno",
    vmin=0.0,
    vmax=vmax_P,
    aspect="auto",
)

ax.set_title(
    f"TPM, median + {int(CI * 100)}% interval "
    f"(stationary probability > {THRESHOLD_PROB:.2f})"
)

ax.set_xlabel(
    "to state"
)

ax.set_ylabel(
    "from state"
)

ax.set_xticks(
    np.arange(m)
)

ax.set_yticks(
    np.arange(m)
)

ax.set_xticklabels(
    [
        str(int(state))
        for state
        in selected_states
    ],
    rotation=90,
)

ax.set_yticklabels(
    [
        str(int(state))
        for state
        in selected_states
    ]
)

cbar = fig.colorbar(
    im,
    ax=ax,
)

cbar.set_label(
    "P$_{ij}$ per 40 ns lag step"
)

norm = im.norm
cmap = im.get_cmap()

for i in range(m):
    for j in range(m):

        val = float(
            P_center[i, j]
        )

        err_lo = float(
            P_err_lo[i, j]
        )

        err_hi = float(
            P_err_hi[i, j]
        )

        rgba = cmap(
            norm(val)
        )

        lum = (
            0.299 * rgba[0]
            + 0.587 * rgba[1]
            + 0.114 * rgba[2]
        )

        txt_color = (
            "white"
            if lum < 0.5
            else "black"
        )

        ax.text(
            j,
            i,
            f"{val:.2g}\n"
            f"+{err_hi:.2g}\n"
            f"-{err_lo:.2g}",
            ha="center",
            va="center",
            color=txt_color,
            fontsize=10,
        )

fig.savefig(
    TPM_OUTPUT_FILE,
    format="pdf",
)

print(
    f"Saved TPM figure to:\n  {TPM_OUTPUT_FILE}"
)

plt.show()


# =============================================================================
# APPROXIMATE RATE MATRIX
#
# Q ≈ (P - I) / lagtime
#
# lagtime = 40 ns
# conversion: 1/ns -> 1/us by multiplying by 1000
# =============================================================================

eye_m = np.eye(
    m,
    dtype=float,
)[None, :, :]

Q_sub_samples_us = (
    (
        P_sub_samples
        - eye_m
    )
    / LAGTIME_NS
) * RATE_SCALE


Q_center = np.quantile(
    Q_sub_samples_us,
    0.5,
    axis=0,
)

Q_lo = np.quantile(
    Q_sub_samples_us,
    Q_LO,
    axis=0,
)

Q_hi = np.quantile(
    Q_sub_samples_us,
    Q_HI,
    axis=0,
)

Q_err_lo = np.maximum(
    Q_center - Q_lo,
    0.0,
)

Q_err_hi = np.maximum(
    Q_hi - Q_center,
    0.0,
)


# =============================================================================
# MASK NEGATIVE GENERATOR DIAGONAL FOR SEQUENTIAL COLOURMAP
# =============================================================================

Q_plot = Q_center.copy()
Q_lo_plot = Q_err_lo.copy()
Q_hi_plot = Q_err_hi.copy()

np.fill_diagonal(
    Q_plot,
    np.nan,
)

np.fill_diagonal(
    Q_lo_plot,
    np.nan,
)

np.fill_diagonal(
    Q_hi_plot,
    np.nan,
)


# =============================================================================
# RATE-MATRIX HEATMAP
# =============================================================================

vmax_Q = float(
    np.nanpercentile(
        Q_plot,
        95,
    )
)

vmax_Q = max(
    vmax_Q,
    1e-12,
)

fig, ax = plt.subplots(
    figsize=(9, 8),
    constrained_layout=True,
    dpi=300,
)

im = ax.imshow(
    np.ma.masked_invalid(
        Q_plot
    ),
    cmap="inferno",
    vmin=0.0,
    vmax=vmax_Q,
    aspect="auto",
)

ax.set_title(
    f"Approximate rate matrix, median + {int(CI * 100)}% interval"
)

ax.set_xlabel(
    "to state"
)

ax.set_ylabel(
    "from state"
)

ax.set_xticks(
    np.arange(m)
)

ax.set_yticks(
    np.arange(m)
)

ax.set_xticklabels(
    [
        str(int(state))
        for state
        in selected_states
    ],
    rotation=90,
)

ax.set_yticklabels(
    [
        str(int(state))
        for state
        in selected_states
    ]
)

cbar = fig.colorbar(
    im,
    ax=ax,
)

cbar.set_label(
    "Q$_{ij}$ (µs$^{-1}$)"
)

norm = im.norm
cmap = im.get_cmap()

for i in range(m):
    for j in range(m):

        if i == j:
            continue

        val = float(
            Q_plot[i, j]
        )

        err_lo = float(
            Q_lo_plot[i, j]
        )

        err_hi = float(
            Q_hi_plot[i, j]
        )

        if not (
            np.isfinite(val)
            and np.isfinite(err_lo)
            and np.isfinite(err_hi)
        ):
            continue

        rgba = cmap(
            norm(val)
        )

        lum = (
            0.299 * rgba[0]
            + 0.587 * rgba[1]
            + 0.114 * rgba[2]
        )

        txt_color = (
            "white"
            if lum < 0.5
            else "black"
        )

        ax.text(
            j,
            i,
            f"{val:.2g}\n"
            f"+{err_hi:.2g}\n"
            f"-{err_lo:.2g}",
            ha="center",
            va="center",
            color=txt_color,
            fontsize=10,
        )

fig.savefig(
    RATE_OUTPUT_FILE,
    format="pdf",
)

print(
    f"Saved rate-matrix figure to:\n  {RATE_OUTPUT_FILE}"
)

plt.show()